# REHAB24-6: 2D / 3D elbow-angle benchmark

Evaluates MediaPipe elbow-angle estimation against the REHAB24-6 OptiTrack 3D
ground truth, and then corrects it.

## Variants

| | method |
|---|---|
| `A_2D_Base` | plain 2D projected angle |
| `B_3D_World` | MediaPipe `worldLandmarks`, ordered joints (paper Eq. 2) |
| `D_2D_Calibrated` / `D_3D_Calibrated` | monotonic calibration curve, leave-one-person-out |
| `D_2D_PerView` | the same curve fitted per camera viewpoint |
| `E_2D_Lifted` / `F_Lift_PerView` | bone-length-constrained 2.5D lift |
| `G_Hybrid` / `H_Hybrid_PerView` | 2D in-plane, lift out-of-plane, blended by planarity |
| `I_ML_Features` | learned geometric correction (Maeda et al. 2026) |

## Inclusion criteria

After Aguilar-Ortega et al., *Sensors* **2023**, 23, 8862, p.10, a recording is
scored only if it has expected flexion at the measured joint (ROM >= 60 deg),
an arm visible in >= 50% of frames, and temporal alignment r >= 0.7. Every
variant is aligned with one lag, estimated once per recording, so the
comparison between variants is like for like.

## Filtering

All variants use a **causal** Butterworth filter (`CAUSAL = True` in Step 5).
`filtfilt` runs forward and backward and so needs the whole clip; reporting
accuracy obtained that way would overstate what a live system can reach.

**Run the cells in order.**

## 1 — Environment and data

Step 2 must run before Step 3: Step 3 uses `DATA`, `glob`, `shutil` and
`zipfile`, all of which Step 2 defines. In the previous version of this
notebook these two were the other way round, so a fresh run failed on a
`NameError`.

In [3]:
# ============================================================
# STEP 1 — Python 3.12 environment for MediaPipe
# ============================================================
import os, subprocess

MP_PY = '/content/mpenv/bin/python'
if not os.path.exists(MP_PY):
    print('Installing uv and creating Python 3.12 env...')
    subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
    subprocess.run(['uv', 'venv', '--python', '3.12', '/content/mpenv'], check=True)
    subprocess.run([
        'uv', 'pip', 'install', '--python', MP_PY,
        'numpy==1.26.4', 'mediapipe==0.10.14', 'pyyaml',
        'pandas', 'opencv-python', 'matplotlib',
        'scikit-learn', 'scipy',
    ], check=True)
print('MediaPipe Python 3.12 env ready at:', MP_PY)

MediaPipe Python 3.12 env ready at: /content/mpenv/bin/python


In [9]:
# ============================================================
# STEP 2 — Settings, Drive, repo clone, recordings list
# ============================================================
import glob, os, shutil, subprocess, sys, time, zipfile
import numpy as np, pandas as pd

# reset working directory FIRST — prevents "No such file or directory" error
os.chdir('/content')

# ---- core settings ----
REPO_DIR         = '/content/repo'
BRANCH           = 'main'
REPO_URL         = 'https://github.com/suvini2001/AI-based-Physiotherapy-project-.git'
MP_PY            = '/content/mpenv/bin/python'
DATA             = '/content/rehab24'
MAX_VIDEOS       = None
EXERCISES        = [1, 2, 3]
THRESHOLD        = 0.70
MARGIN           = 45
MAX_LAG          = 30
MODEL_COMPLEXITY = 1

ARMS_OVERRIDE = {
    'right': ('12: RightArm', '13: RightForeArm', '14: RightHand'),
    'left':  ('7: LeftArm',   '8: LeftForeArm',   '9: LeftHand'),
}

# ---- mount Drive ----
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
CACHE   = '/content/drive/MyDrive/rehab24_cache_3d'
RESULTS = '/content/drive/MyDrive/rehab24_results_3d'
for d in (DATA, CACHE, RESULTS):
    os.makedirs(d, exist_ok=True)

# ---- remove any old clone and clone correct branch ----
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
assert not os.path.exists(REPO_DIR), "Could not remove old repo"
subprocess.run(
    ['git', 'clone', '-q', '-b', BRANCH, REPO_URL, REPO_DIR],
    check=True, cwd='/content'
)
os.chdir(REPO_DIR)
for p in (REPO_DIR,
          os.path.join(REPO_DIR, 'capture'),
          os.path.join(REPO_DIR, 'pose'),
          os.path.join(REPO_DIR, 'vision')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Branch:', BRANCH)

# ---- verify modules ----
import importlib.util
all_ok = True
for mod in ['angles', 'angles_world', 'benchmark_common',
            'calibration', 'filters', 'visibility']:
    spec = importlib.util.find_spec(mod)
    status = 'OK' if spec else 'MISSING!!!'
    if not spec: all_ok = False
    print(f'  {mod:20s}: {status}')
assert all_ok, "Some modules missing — check branch"

# ---- joint names ----
jn_files = glob.glob(f'{DATA}/**/*joint*names*.txt', recursive=True)
assert jn_files, "joints_names.txt not found — re-run Step 3 first"
with open(jn_files[0]) as f:
    JOINT_NAMES = [l.strip() for l in f if l.strip()]
print(f'Joint names: {len(JOINT_NAMES)} loaded')

for side, joints in ARMS_OVERRIDE.items():
    for jn in joints:
        assert jn in JOINT_NAMES, f"'{jn}' not in JOINT_NAMES"
print('ARMS_OVERRIDE: all joints verified OK')

# ---- segmentation ----
seg_files = glob.glob(f'{DATA}/**/Segmentation.*', recursive=True)
assert seg_files, "Segmentation.csv not found — re-run Step 3 first"
sep = ';' if ';' in open(seg_files[0]).readline() else ','
seg_df = pd.read_csv(seg_files[0], sep=sep)
cols_lower = {c.lower(): c for c in seg_df.columns}

def pick_col(candidates):
    for cand in candidates:
        for cl, orig in cols_lower.items():
            if cand in cl: return orig
    raise KeyError(f'No column matching {candidates}')

c_vid   = pick_col(['video_id', 'video', 'file', 'rec', 'name'])
c_ex    = pick_col(['exercise_id', 'exercise', 'activity', 'ex'])
c_start = pick_col(['first_frame', 'start', 'begin', 'f0', 'from'])
c_end   = pick_col(['last_frame',  'end',   'stop',  'f1', 'to'])
c_view  = None
for cl, orig in cols_lower.items():
    if any(k in cl for k in ['cam17_orientation', 'view', 'angle', 'cam_pos']):
        c_view = orig
        break
print(f'Columns: vid={c_vid}, ex={c_ex}, start={c_start}, end={c_end}, view={c_view}')

# ---- index files ----
all_videos = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.mp4', '.avi', '.mov'))
}
SKIP = {'joints_names.txt', 'joint_names.txt',
        'segmentation.csv', 'segmentation.txt'}
all_3d = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.npy', '.npz'))
    and os.path.basename(p).lower() not in SKIP
}
print(f'Videos: {len(all_videos)},  3D truth files: {len(all_3d)}')

# ---- build recordings ----
seg_ex = seg_df[seg_df[c_ex].isin(EXERCISES)].copy()
recordings = []
for (vid_id, ex_id), grp in seg_ex.groupby([c_vid, c_ex]):
    person_id = str(vid_id).strip()
    f0 = max(0, int(grp[c_start].min()) - MARGIN)
    f1 = int(grp[c_end].max()) + MARGIN
    view_val = str(grp[c_view].iloc[0]) if c_view else 'unknown'
    matched_vids = {k: v for k, v in all_videos.items()
                    if k.startswith(person_id + '-') or k == person_id}
    person_3d = {k: v for k, v in all_3d.items()
                 if k.startswith(person_id + '-') or k == person_id}
    if not person_3d: continue
    truth_key = next((k for k in person_3d if '30fps' in k),
                    next((k for k in person_3d if '120fps' in k), next(iter(person_3d))))
    for vid_key, vid_path in matched_vids.items():
        cam = ('cam18'  if 'cam18' in vid_key.lower() or 'transposed' in vid_key.lower()
               else 'cam17' if 'cam17' in vid_key.lower()
               else 'unknown')
        recordings.append({
            'rec_id':     f'{person_id}_ex{int(ex_id)}_{cam}',
            'video_key':  vid_key,
            'video_path': vid_path,
            'truth_path': person_3d[truth_key],
            'truth_key':  truth_key,
            'exercise':   int(ex_id),
            'f0': f0, 'f1': f1,
            'view_raw':   view_val,
            'cam':        cam,
            'cam_label':  ('cam18' if 'camera18' in vid_key.lower() or 'cam18' in vid_key.lower()
                           else 'cam17' if 'camera17' in vid_key.lower() or 'cam17' in vid_key.lower()
                           else 'unknown'),
        })

recordings = list({r['rec_id']: r for r in recordings}.values())
if MAX_VIDEOS:
    recordings = recordings[:MAX_VIDEOS]
print(f'Total recordings: {len(recordings)}')

# ---- cache status ----
cached_files = {os.path.basename(p) for p in glob.glob(f'{CACHE}/*.npz')}
matched = sum(1 for r in recordings
              if f"{r['rec_id']}_{r['f0']}_{r['f1']}_3d.npz" in cached_files)
print(f'Cache: {matched}/{len(recordings)} done — {len(recordings)-matched} remaining')
print('\nReady — now run Step 3')

Mounted at /content/drive
Branch: main
  angles              : OK
  angles_world        : OK
  benchmark_common    : OK
  calibration         : OK
  filters             : OK
  visibility          : OK
Joint names: 26 loaded
ARMS_OVERRIDE: all joints verified OK
Columns: vid=video_id, ex=exercise_id, start=first_frame, end=last_frame, view=cam17_orientation
Videos: 130,  3D truth files: 130
Total recordings: 70
Cache: 70/70 done — 0 remaining

Ready — now run Step 3


In [10]:
# ============================================================
# STEP 3 — Locate the dataset in Drive, or fetch it from Zenodo
# ============================================================
import urllib.request

ZENODO_BASE = 'https://zenodo.org/records/13305826/files'

def find_in_drive(filename):
    if not os.path.exists('/content/drive/MyDrive'):
        return None
    matches = glob.glob(f'/content/drive/MyDrive/**/{filename}', recursive=True)
    return matches[0] if matches else None

def ensure_file(filename, url_name=None):
    target = os.path.join(DATA, filename)
    if os.path.exists(target) and os.path.getsize(target) > 0:
        print(f'[OK]       {filename}')
        return target
    hit = find_in_drive(filename)
    if hit:
        print(f'[Drive]    {filename} <- {hit}')
        if not os.path.exists(target):
            os.symlink(hit, target)
        return target
    url = f"{ZENODO_BASE}/{url_name or filename}?download=1"
    print(f'[Download] {url}')
    urllib.request.urlretrieve(url, target)
    return target

# --- Segmentation.csv ---
for name in ('Segmentation.csv', 'Segmentation.txt'):
    hit = find_in_drive(name)
    if hit:
        dst = os.path.join(DATA, os.path.basename(hit))
        if not os.path.exists(dst):
            shutil.copy(hit, dst)
        break
if not glob.glob(f'{DATA}/Segmentation.*'):
    try: ensure_file('Segmentation.csv')
    except: ensure_file('Segmentation.txt')

# --- joints_names.txt ---
for name in ('joints_names.txt', 'joint_names.txt'):
    hit = find_in_drive(name)
    if hit:
        dst = os.path.join(DATA, os.path.basename(hit))
        if not os.path.exists(dst):
            shutil.copy(hit, dst)
        break
if not glob.glob(f'{DATA}/joint*names*.txt'):
    ensure_file('joints_names.txt')

# --- videos (Ex1..Ex6 structure OR videos.zip) ---
# First check if videos already extracted into Ex* folders
existing_mp4s = glob.glob(f'{DATA}/**/*.mp4', recursive=True) + \
                glob.glob(f'{DATA}/**/*.avi', recursive=True)
if not existing_mp4s:
    # try Drive zip
    drive_zip = find_in_drive('videos.zip')
    if drive_zip:
        target = os.path.join(DATA, 'videos.zip')
        if not os.path.exists(target):
            os.symlink(drive_zip, target)
        print('Extracting videos.zip...')
        with zipfile.ZipFile(target) as zf:
            zf.extractall(DATA)
    else:
        z = ensure_file('videos.zip')
        print('Extracting videos.zip...')
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA)

# --- 3d_joints ---
existing_3d = glob.glob(f'{DATA}/3d_joints/**/*.*', recursive=True)
if not existing_3d:
    drive_3d_dir = None
    hits = glob.glob('/content/drive/MyDrive/**/3d_joints', recursive=True)
    if hits and os.path.isdir(hits[0]) and os.listdir(hits[0]):
        drive_3d_dir = hits[0]
    if drive_3d_dir:
        dst = os.path.join(DATA, '3d_joints')
        if not os.path.exists(dst):
            os.symlink(drive_3d_dir, dst)
        print(f'[Drive] Linked 3d_joints from {drive_3d_dir}')
    else:
        z = ensure_file('3d_joints.zip')
        print('Extracting 3d_joints.zip...')
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA)

print('\nDATA contents:', os.listdir(DATA))
all_mp4s = glob.glob(f'{DATA}/**/*.mp4', recursive=True) + \
           glob.glob(f'{DATA}/**/*.avi', recursive=True)
print(f'Total video files found: {len(all_mp4s)}')
if all_mp4s:
    print('  Example:', all_mp4s[0])

[OK]       3d_joints.zip
Extracting 3d_joints.zip...

DATA contents: ['3d_joints.zip', 'Ex4', 'Ex6', 'Ex2', 'videos.zip', 'Ex5', 'Segmentation.csv', 'joints_names.txt', 'Ex1', 'Ex3']
Total video files found: 130
  Example: /content/rehab24/Ex4/PM_041-Camera17-30fps.mp4


## 2 — Which recordings can be scored at all

The paper restricts angle evaluation to exercises with expected flexion at
the measured joint. This scan shows the split is clean rather than
arbitrary: Ex1 tops out around 51 deg, Ex2 and Ex3 start above 80 deg, so a
60 deg threshold separates them with nothing near the boundary.

In [11]:
# ============================================================
# ROM SCAN — which recordings actually flex the elbow?
# (paper p.10: "only ... exercises where there is an expected flexion")
# ============================================================
import numpy as np, pandas as pd
import benchmark_common as bc

rows = []
for rec in recordings:
    truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
    for side in ('right', 'left'):
        gt = bc.truth_elbow_3d(truth_arr, JOINT_NAMES, ARMS_OVERRIDE[side], use_dims=3)
        if np.isfinite(gt).any():
            rom = float(np.nanpercentile(gt, 95) - np.nanpercentile(gt, 5))
            rows.append((rec['exercise'], side, rec['rec_id'], rom))

df_rom = pd.DataFrame(rows, columns=['ex', 'side', 'rec_id', 'rom'])

MIN_ROM = 60.0        # must match the gate in Step 5
print(f'usable fraction (ROM >= {MIN_ROM:.0f} deg):')
print(df_rom.assign(ok=df_rom.rom >= MIN_ROM).groupby(['ex','side'])['ok'].mean().round(2))
print()
print('TOTAL usable:', int((df_rom.rom >= MIN_ROM).sum()), '/', len(df_rom))

usable fraction (ROM >= 60 deg):
ex  side 
1   left     0.0
    right    0.0
2   left     1.0
    right    1.0
3   left     1.0
    right    1.0
Name: ok, dtype: float64

TOTAL usable: 88 / 140


## 3 — The benchmark

One pass over every recording. For each arm it builds all four uncalibrated
variants, applies the three inclusion gates, aligns once, and only then
scores — so no metric is ever computed on unaligned signals. The calibrated
variants are fitted afterwards, leaving out the person being scored.

The per-frame geometric feature matrix is built here too, from the same
signals the benchmark scores and aligned with the same lag, so Step 6
cannot drift away from it.

In [12]:
# ============================================================
# STEP 5 — Evaluation (A / B / E / G, then calibrated D / F / H)
# ============================================================
import numpy as np, os, subprocess, time, csv
import pandas as pd
from features import frame_features, overlap_rows, FEATURE_NAMES

import angles as angles_2d
from angles_lift import bone_lengths, elbow_angle_lifted, elbow_angle_hybrid, planarity
from angles_world import mask_world_like
import benchmark_common as bc
from benchmark_common import elbow_angle_3d_ordered
from benchmark_common import butterworth_filter
from visibility import mask_low_visibility
from calibration import fit_curve, apply_calibration, leave_one_person_out

# ------------------------------------------------------------------
# CAUSAL = True means every variant is filtered the way a live system
# would have to filter it.
#
# scipy filtfilt runs the filter forward AND backward, so it needs the
# whole clip. No real-time system can reproduce it: measured on a real
# frontal recording, an unfiltered live angle sat a mean of 0.21 deg and
# a maximum of 20.6 deg away from the filtfilt series. Since angle_2d
# carries ~41% of the learned model's gain, evaluating with filtfilt
# would report an accuracy the deployed system cannot reach.
#
# It also has to be the same for every variant: the temporal lag is
# estimated once from pred_a and applied to all of them, so filtering
# one causally and the rest non-causally would misalign the others by
# the filter's group delay. Set CAUSAL = False only to reproduce the
# older offline numbers.
# ------------------------------------------------------------------
CAUSAL = True


def fill_1d(arr, max_gap=15):
    arr = np.array(arr, dtype=float)
    if np.isnan(arr).all() or not np.isnan(arr).any():
        return arr
    idx  = np.arange(len(arr))
    good = ~np.isnan(arr)
    filled = arr.copy()
    in_gap, gap_start = False, 0
    for i in range(len(arr)):
        if np.isnan(arr[i]) and not in_gap:
            in_gap, gap_start = True, i
        elif not np.isnan(arr[i]) and in_gap:
            in_gap = False
            if i - gap_start <= max_gap:
                filled[gap_start:i] = np.interp(
                    idx[gap_start:i], idx[good], arr[good])
    return filled


def infer_view(rec):
    """cam17_orientation describes the SUBJECT's orientation w.r.t. camera 17.
    It must not be reused as camera 18's viewpoint."""
    v   = str(rec.get('view_raw', '')).lower()
    cam = rec.get('cam_label', rec.get('cam', ''))

    if 'half' in v:                   subj = 'Half-profile'
    elif 'front' in v or v == '0':    subj = 'Front'
    elif 'prof' in v:                 subj = 'Profile'
    elif 'side' in v or '90' in v:    subj = 'Side'
    else:                             subj = 'Unknown'

    if cam == 'cam18':
        return f'{subj}/cam18'        # same subject pose, different camera
    return subj


def get_3d_landmarks(rec):
    cache_path = os.path.join(
        CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz")
    need_extract = True
    if os.path.exists(cache_path):
        try:
            d = np.load(cache_path)
            if 'world' in d.files and 'lm' in d.files and d['lm'].shape[0] > 0:
                need_extract = False
        except Exception:
            pass
    if need_extract:
        env = os.environ.copy()
        env['MPLBACKEND'] = 'Agg'
        env['PYTHONPATH'] = REPO_DIR
        cmd = [MP_PY, 'extract_world.py',
               rec['video_path'],
               str(rec['f0']), str(rec['f1']),
               cache_path, str(MODEL_COMPLEXITY)]
        subprocess.run(cmd, check=True, env=env, cwd=REPO_DIR)
    d = np.load(cache_path)
    return d['lm'], d['world'], tuple(d['size']), float(d['fps'])


def resample_to(t_target, t_src, values, max_gap_s=0.1):
    """Linear-interpolate 1-D values onto t_target. NaN-safe: a real mocap
    dropout stays NaN instead of being interpolated across."""
    values = np.asarray(values, float)
    good = np.isfinite(values)
    if good.sum() < 2:
        return np.full(len(t_target), np.nan)
    out = np.interp(t_target, t_src[good], values[good], left=np.nan, right=np.nan)
    if max_gap_s is not None:
        idx = np.searchsorted(t_src[good], t_target)
        idx = np.clip(idx, 1, good.sum() - 1)
        near = np.minimum(np.abs(t_target - t_src[good][idx]),
                          np.abs(t_target - t_src[good][idx - 1]))
        out[near > max_gap_s] = np.nan
    return out


def truth_angle_for(rec, truth_arr, world, npz_f0, fps, arm_joints):
    """gt angle on the VIDEO frame grid, whatever rate the truth file is."""
    TRUTH_FPS = 120.0 if '120fps' in str(rec.get('truth_key', '')) else 30.0
    if abs(TRUTH_FPS - fps) < 1.0:
        win = truth_arr[npz_f0: npz_f0 + len(world)]
        return bc.truth_elbow_3d(win, JOINT_NAMES, arm_joints, use_dims=3)
    gt_full = bc.truth_elbow_3d(truth_arr, JOINT_NAMES, arm_joints, use_dims=3)
    t_truth = np.arange(len(truth_arr)) / TRUTH_FPS
    t_video = (npz_f0 + np.arange(len(world))) / fps
    return resample_to(t_video, t_truth, gt_full)


def append_row(path, row):
    """Append one row dict to a CSV, creating the header if the file is new."""
    write_header = not os.path.exists(path) or os.path.getsize(path) == 0
    with open(path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            writer.writeheader()
        writer.writerow(row)


# ---- one CSV per uncalibrated variant; the calibrated ones share path_D ----
path_A = os.path.join(RESULTS, 'variant_A_2d.csv')
path_B = os.path.join(RESULTS, 'variant_B_3d_world.csv')
path_D = os.path.join(RESULTS, 'variant_D_3d_calibrated.csv')
path_E = os.path.join(RESULTS, 'variant_E_lifted.csv')
path_G = os.path.join(RESULTS, 'variant_G_hybrid.csv')

# fresh start: append_row only appends, so stale rows would accumulate
for path in (path_A, path_B, path_D, path_E, path_G):
    if os.path.exists(path):
        os.remove(path)

skipped_align  = 0
skipped_static = 0
skipped_cov    = 0
MIN_ROM = 60.0        # Ex1 max 50.8 deg, Ex2/Ex3 min 80.4 deg -> clean split
MIN_COV = 50.0        # % of frames needing a usable 2D angle

rows_A, rows_B, rows_E, rows_G, aligned_cache = [], [], [], [], []

for idx, rec in enumerate(recordings, 1):
    try:
        t0 = time.time()
        lm, world, size, fps = get_3d_landmarks(rec)
        truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)

        npz_f0 = int(np.load(os.path.join(
            CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz"))['f0'])
        TRUTH_FPS = 120.0 if "120fps" in str(rec.get("truth_key", "")) else 30.0
        if abs(TRUTH_FPS - fps) < 1.0:
            truth_win = truth_arr[npz_f0: npz_f0 + len(world)]   # rates match
            t_video = t_truth = None
        else:
            truth_win = None                                      # interp needed
            t_truth = np.arange(len(truth_arr)) / TRUTH_FPS
            t_video = (npz_f0 + np.arange(len(world))) / fps

        lm_masked    = mask_low_visibility(lm, threshold=THRESHOLD)
        world_masked = mask_world_like(world, lm_masked)

        # bone lengths are a property of the person, so once per recording
        L = {s: bone_lengths(lm_masked, s, size) for s in ('right', 'left')}
        side_mae = {'right': float('nan'), 'left': float('nan')}

        for side in ('right', 'left'):
            arm_joints = ARMS_OVERRIDE[side]
            if truth_win is not None:
                gt_3d = bc.truth_elbow_3d(truth_win, JOINT_NAMES, arm_joints, use_dims=3)
            else:
                gt_full = bc.truth_elbow_3d(truth_arr, JOINT_NAMES, arm_joints, use_dims=3)
                gt_3d = resample_to(t_video, t_truth, gt_full)

            L_up, L_fo = L[side]

            # --- A: plain 2D projection ---
            pred_a_raw  = angles_2d.elbow_angle(lm_masked, side=side, size=size)
            pred_a_fill = fill_1d(pred_a_raw, max_gap=5)
            pred_a = (butterworth_filter(pred_a_fill, fs=fps, cutoff=6.0, causal=CAUSAL)
                      if np.isfinite(pred_a_fill).sum() > 15 else pred_a_fill)

            # --- B: MediaPipe world landmarks ---
            pred_b_raw  = elbow_angle_3d_ordered(world_masked, side=side)
            pred_b_fill = fill_1d(pred_b_raw, max_gap=5)
            pred_b = (butterworth_filter(pred_b_fill, fs=fps, cutoff=6.0, causal=CAUSAL)
                      if np.isfinite(pred_b_fill).sum() > 15 else pred_b_fill)

            # --- E: bone-length constrained 2.5D lift ---
            pred_e_raw  = elbow_angle_lifted(lm_masked, world_masked, side, size,
                                             L_up, L_fo)
            pred_e_fill = fill_1d(pred_e_raw, max_gap=5)
            pred_e = (butterworth_filter(pred_e_fill, fs=fps, cutoff=6.0, causal=CAUSAL)
                      if np.isfinite(pred_e_fill).sum() > 15 else pred_e_fill)

            # --- G: hybrid, 2D in-plane and lift out-of-plane ---
            pred_g_raw  = elbow_angle_hybrid(lm_masked, world_masked, side, size,
                                             L_up, L_fo)
            pred_g_fill = fill_1d(pred_g_raw, max_gap=5)
            pred_g = (butterworth_filter(pred_g_fill, fs=fps, cutoff=6.0, causal=CAUSAL)
                      if np.isfinite(pred_g_fill).sum() > 15 else pred_g_fill)

            # per-frame geometric features, same signals the benchmark scores
            feat_full = frame_features(lm_masked, world_masked, side, size,
                                       L_up, L_fo, pred_a, pred_b, pred_e)

            meta = {
                'recording': f"{rec['rec_id']}_{side}",
                'video':     rec['video_key'],
                'person':    bc.person_of(rec['video_key']),
                'ex':        rec['exercise'],
                'cam':       rec.get('cam_label', rec.get('cam', 'unknown')),
                'side':      side,
                'view':      infer_view(rec),
            }

            # --- gate 1: does this joint actually flex? (paper p.10) ---
            gt_range = (float(np.nanpercentile(gt_3d, 95) - np.nanpercentile(gt_3d, 5))
                        if np.isfinite(gt_3d).any() else float('nan'))
            meta = dict(meta, gt_range=round(gt_range, 1) if gt_range == gt_range else None)

            if not (gt_range >= MIN_ROM):
                skipped_static += 1
                print(f"    static {side}: ROM {gt_range:.1f}deg < {MIN_ROM:.0f} "
                      f"- no expected flexion")
                continue

            # --- gate 1b: is this arm visible at all? ---
            cov_a = 100.0 * float(np.isfinite(pred_a).mean())
            meta = dict(meta, cov_pred=round(cov_a, 1))

            if cov_a < MIN_COV:
                skipped_cov += 1
                print(f"    occluded {side}: coverage {cov_a:.1f}% "
                      f"(ROM {gt_range:.0f}deg is fine) - arm hidden from this camera")
                continue

            # --- gate 2: temporal alignment, one lag for every variant ---
            lag, lag_r = bc.best_lag(pred_a, gt_3d, max_lag=MAX_LAG, filter_signals=True)
            pa_al, gt_al = bc.overlap(pred_a, gt_3d, lag)
            pb_al, _     = bc.overlap(pred_b, gt_3d, lag)
            pe_al, _     = bc.overlap(pred_e, gt_3d, lag)
            pg_al, _     = bc.overlap(pred_g, gt_3d, lag)
            feat_al, _   = overlap_rows(feat_full, gt_3d, lag)
            meta = dict(meta, lag=lag, lag_r=round(float(lag_r), 3))

            if abs(lag) >= MAX_LAG or lag_r < 0.7:
                skipped_align += 1
                why = "lag pinned" if abs(lag) >= MAX_LAG else "weak tracking"
                print(f"    reject {side}: {why}  lag={lag} r={lag_r:.2f} "
                      f"ROM={gt_range:.0f}deg cov={cov_a:.0f}%")
                continue

            # every row is built after the gate, from the same aligned signals
            row_a = bc.make_row("A_2D_Base",   pa_al, gt_al, meta)
            row_b = bc.make_row("B_3D_World",  pb_al, gt_al, meta)
            row_e = bc.make_row("E_2D_Lifted", pe_al, gt_al, meta)
            row_g = bc.make_row("G_Hybrid",    pg_al, gt_al, meta)

            if row_b:
                rows_B.append(row_b)
                append_row(path_B, row_b)
                side_mae[side] = row_b["mae"]
            if row_e:
                rows_E.append(row_e)
                append_row(path_E, row_e)
            if row_g:
                rows_G.append(row_g)
                append_row(path_G, row_g)
            if row_a:
                rows_A.append(row_a)
                append_row(path_A, row_a)
                aligned_cache.append({"meta":      meta,
                                      "pred_2d":   pa_al,
                                      "pred_3d":   pb_al,
                                      "pred_lift": pe_al,
                                      "pred_hyb":  pg_al,
                                      "feat":      feat_al,
                                      "truth_3d":  gt_al})

        elapsed = time.time() - t0
        mae_r, mae_l = side_mae['right'], side_mae['left']
        print(f'[{idx}/{len(recordings)}] {rec["rec_id"]}  '
              f'{elapsed:.1f}s  MAE R/L: {mae_r:.1f}deg/{mae_l:.1f}deg')

    except Exception as e:
        import traceback
        print(f'[{idx}/{len(recordings)}] SKIPPED {rec["rec_id"]}: {e}')
        traceback.print_exc()


print(f"\nSkipped (no flexion, ROM < {MIN_ROM:.0f}deg)  : {skipped_static}")
print(f"Skipped (occluded, cov < {MIN_COV:.0f}%)       : {skipped_cov}")
print(f"Skipped (poor tracking, r < 0.7)      : {skipped_align}")
print(f"Kept                                  : {len(rows_A)}")

# ---- calibrated variants: leave-one-person-out, global and per-view ----
rows_D = []
for item in aligned_cache:
    me   = item['meta']
    pool = [x for x in aligned_cache if x['meta']['person'] != me['person']]
    if not pool:
        continue
    same_view = [x for x in pool if x['meta']['view'] == me['view']]

    jobs = [('pred_2d',   pool,      'D_2D_Calibrated'),
            ('pred_3d',   pool,      'D_3D_Calibrated'),
            ('pred_2d',   same_view, 'D_2D_PerView'),
            ('pred_lift', same_view, 'F_Lift_PerView'),
            ('pred_hyb',  same_view, 'H_Hybrid_PerView')]

    for key, train, label in jobs:
        if label.endswith('PerView') and len(train) < 3:
            train = pool                      # fallback: too little same-view data
        pred_cal = item[key]
        p_cat = np.concatenate([x[key]        for x in train])
        t_cat = np.concatenate([x['truth_3d'] for x in train])
        valid = np.isfinite(p_cat) & np.isfinite(t_cat)
        if valid.sum() > 20:
            try:
                params = fit_curve(p_cat[valid], t_cat[valid], min_count=15)
                pred_cal = apply_calibration(item[key], params)
            except ValueError as e:
                print(f'  calib skip ({label}):', e)
        row = bc.make_row(label, pred_cal, item['truth_3d'], dict(me))
        if row:
            rows_D.append(row)
            append_row(path_D, row)

print(f'\nDone. Rows: A={len(rows_A)}, B={len(rows_B)}, '
      f'E={len(rows_E)}, G={len(rows_G)}, D={len(rows_D)}')

    static right: ROM 30.3deg < 60 - no expected flexion
    static left: ROM 2.5deg < 60 - no expected flexion
[1/70] PM_000_ex1_unknown  0.5s  MAE R/L: nandeg/nandeg


/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/content/repo/angles_lift.py:238: RuntimeWarning: All-NaN slice encountered
  out = np.nanmedian(stack, axis=0)


    static right: ROM 30.3deg < 60 - no expected flexion
    static left: ROM 2.5deg < 60 - no expected flexion
[2/70] PM_000_ex1_cam18  1.3s  MAE R/L: nandeg/nandeg
    static right: ROM 11.7deg < 60 - no expected flexion
    static left: ROM 3.7deg < 60 - no expected flexion
[3/70] PM_001_ex1_unknown  1.0s  MAE R/L: nandeg/nandeg
    static right: ROM 11.7deg < 60 - no expected flexion
    static left: ROM 3.7deg < 60 - no expected flexion
[4/70] PM_001_ex1_cam18  1.1s  MAE R/L: nandeg/nandeg
    static right: ROM 13.5deg < 60 - no expected flexion
    static left: ROM 2.1deg < 60 - no expected flexion
[5/70] PM_002_ex1_unknown  1.1s  MAE R/L: nandeg/nandeg
    static right: ROM 13.5deg < 60 - no expected flexion
    static left: ROM 2.1deg < 60 - no expected flexion
[6/70] PM_002_ex1_cam18  1.1s  MAE R/L: nandeg/nandeg
[7/70] PM_003_ex2_unknown  1.3s  MAE R/L: 15.0deg/10.9deg
    occluded right: coverage 0.0% (ROM 105deg is fine) - arm hidden from this camera
[8/70] PM_003_ex2_cam18

## 4 — Learned geometric correction

In [14]:
# ==================================================================
# STEP 6 — I_ML_Features : learned geometric correction
#
#   Method after Maeda et al., BMC Musculoskelet Disord 2026, 27:552:
#   convert the MediaPipe skeleton into per-frame geometric features and
#   regress the true joint angle from them, instead of mapping one
#   measured angle through a calibration curve. A curve can only remove a
#   bias that depends on the angle; a feature set that also describes HOW
#   the arm sits relative to the camera lets the model learn the
#   projection distortion itself, which is what varies between viewpoints.
#
#   Two decisions, both forced by the diagnostics further down:
#
#     1. The four constant-per-recording features are DROPPED.
#        'bone_upper', 'bone_fore', 'sw_frac' and 'is_right' hold one
#        value for a whole recording, so a tree can use them as a
#        recording fingerprint. Check A shows the accuracy difference is
#        within the fold-to-fold spread, so this is a transferability
#        decision, not an accuracy one.
#
#     2. A frame with NaN features is KEPT.
#        Dropping a frame because ANY feature is NaN throws away frames
#        whose features are missing by construction (d_angle on frame 0,
#        plan_* when a bone length could not be estimated). LightGBM
#        learns a default direction for missing values, so a frame is
#        kept whenever the measured 2D angle and the ground truth both
#        exist. What is lost now is only the ~6% of frames where the arm
#        is not measurable at all, which is occlusion, not bookkeeping.
#
#   The feature matrix is NOT rebuilt here. Step 5 already computed it
#   from the same signals it scored, and aligned it with the same lag
#   through features.overlap_rows, so reusing item['feat'] removes any
#   possibility of this cell and Step 5 disagreeing.
# ==================================================================
import os
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

import benchmark_common as bc
from features import FEATURE_NAMES

for _need in ('aligned_cache', 'RESULTS', 'append_row'):
    if _need not in dir():
        raise RuntimeError(f"'{_need}' is not defined — run Step 5 first.")

# ------------------------------------------------------------------
# feature subset
# ------------------------------------------------------------------
DROP = {'bone_upper', 'bone_fore', 'sw_frac', 'is_right'}

KEEP_IDX   = [i for i, n in enumerate(FEATURE_NAMES) if n not in DROP]
KEEP_NAMES = [FEATURE_NAMES[i] for i in KEEP_IDX]
I_ANGLE2D  = FEATURE_NAMES.index('angle_2d')      # index in the FULL matrix

print(f'features kept    : {len(KEEP_NAMES)}  {KEEP_NAMES}')
print(f'features dropped : {len(DROP)}  {sorted(DROP)}')

# ------------------------------------------------------------------
# assemble the training matrix from Step 5's aligned output
# ------------------------------------------------------------------
X_parts, y_parts, g_parts, k_parts = [], [], [], []
FEAT_MASKS = []                 # (trimmed full-20 matrix, mask), for Check A
meta_by_rec = {}
n_raw = n_kept = 0

for item in aligned_cache:
    me = item['meta']
    F = np.asarray(item['feat'], float)
    t = np.asarray(item['truth_3d'], float)
    n = min(len(F), len(t))
    F, t = F[:n], t[:n]

    # keep the frame if the measured angle and the truth exist; every other
    # NaN is handed to LightGBM
    ok = np.isfinite(F[:, I_ANGLE2D]) & np.isfinite(t)
    n_raw += n
    n_kept += int(ok.sum())
    if not ok.any():
        continue

    X_parts.append(F[ok][:, KEEP_IDX].astype(np.float32))
    y_parts.append(t[ok])
    g_parts.append(np.full(int(ok.sum()), me['person']))
    k_parts.append(np.full(int(ok.sum()), me['recording']))
    FEAT_MASKS.append((F, ok))
    meta_by_rec[me['recording']] = me

X       = np.concatenate(X_parts)
y       = np.concatenate(y_parts)
groups  = np.concatenate(g_parts)
rec_key = np.concatenate(k_parts)

print(f'\nframes           : {len(y):,} from {len(meta_by_rec)} arm-recordings')
print(f'coverage         : {100.0 * n_kept / max(n_raw, 1):.2f}% '
      f'({n_kept:,} / {n_raw:,}) — the shortfall is frames with no '
      f'measurable 2D angle')
print(f'NaNs left in X   : {100.0 * np.isnan(X).mean():.2f}% of cells '
      f'(LightGBM splits on these natively)')
print(f'participants     : {len(np.unique(groups))}')

# ------------------------------------------------------------------
# out-of-fold prediction, folds grouped by participant
# ------------------------------------------------------------------
PARAMS = dict(objective='l1',       # train the loss that gets reported
              n_estimators=600, learning_rate=0.05, num_leaves=63,
              min_child_samples=80, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, verbosity=-1)

n_persons = len(np.unique(groups))
if n_persons < 2:
    raise RuntimeError('need at least 2 participants for grouped folds')
N_SPLITS = int(min(5, n_persons))

Xdf_full = pd.DataFrame(X, columns=KEEP_NAMES)
oof = np.full(len(y), np.nan)
fold_mae = []
for fold, (tr, te) in enumerate(
        GroupKFold(n_splits=N_SPLITS).split(Xdf_full, y, groups), 1):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model = lgb.LGBMRegressor(**PARAMS).fit(Xdf_full.iloc[tr], y[tr])
        oof[te] = model.predict(Xdf_full.iloc[te])
    fold_mae.append(float(np.mean(np.abs(oof[te] - y[te]))))
    print(f'  fold {fold}/{N_SPLITS}  held out '
          f'{sorted(set(groups[te]))}  MAE {fold_mae[-1]:.2f} deg')

oof = np.clip(oof, 0.0, 180.0)        # an elbow angle cannot leave [0, 180]

raw_2d = X[:, KEEP_NAMES.index('angle_2d')].astype(float)
fin = np.isfinite(raw_2d)
print(f'\nfold spread      : {min(fold_mae):.2f} to {max(fold_mae):.2f} deg '
      f'— the uncertainty on every number below')
print(f'frame-level MAE  : ML {np.mean(np.abs(oof - y)):.2f} deg  vs  '
      f'raw 2D {np.mean(np.abs(raw_2d[fin] - y[fin])):.2f} deg')
print(f'frame-level r    : ML {np.corrcoef(oof, y)[0, 1]:.3f}  vs  '
      f'raw 2D {np.corrcoef(raw_2d[fin], y[fin])[0, 1]:.3f}')

# ------------------------------------------------------------------
# per-recording rows, so this variant joins the results table
# ------------------------------------------------------------------
path_I = os.path.join(RESULTS, 'variant_I_ml_features.csv')
if os.path.exists(path_I):
    os.remove(path_I)

rows_I = []
for recording, me in meta_by_rec.items():
    sel = rec_key == recording
    row = bc.make_row('I_ML_Features', oof[sel], y[sel], dict(me))
    if row:
        rows_I.append(row)
        append_row(path_I, row)

df_I = pd.DataFrame(rows_I)
print(f"\nI_ML_Features    : {len(rows_I)} rows  "
      f"median MAE {df_I['mae'].median():.2f} deg  "
      f"mean {df_I['mae'].mean():.2f} deg")

print('\nby viewpoint:')
print(df_I.groupby('view')['mae']
      .agg(n='size', median='median', mean='mean').round(2).to_string())
print('\nby exercise:')
print(df_I.groupby('ex')['mae']
      .agg(n='size', median='median', mean='mean').round(2).to_string())

# ------------------------------------------------------------------
# what the model actually leans on
# ------------------------------------------------------------------
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    probe = lgb.LGBMRegressor(**PARAMS).fit(Xdf_full, y)

imp = (pd.DataFrame({'feature': KEEP_NAMES,
                     'gain': probe.booster_.feature_importance('gain')})
       .assign(pct=lambda d: 100.0 * d['gain'] / d['gain'].sum())
       .sort_values('gain', ascending=False))
print('\nfeature importance (gain, %):')
print(imp[['feature', 'pct']].round(2).to_string(index=False))

features kept    : 16  ['angle_2d', 'angle_world', 'angle_lift', 'plan_upper', 'plan_fore', 'plan_min', 'len_upper', 'len_fore', 'vis_sh', 'vis_el', 'vis_wr', 'wz_sh_el', 'wz_el_wr', 'torso_ratio', 'd_angle', 'std_angle']
features dropped : 4  ['bone_fore', 'bone_upper', 'is_right', 'sw_frac']

frames           : 149,973 from 56 arm-recordings
coverage         : 93.53% (149,973 / 160,356) — the shortfall is frames with no measurable 2D angle
NaNs left in X   : 0.16% of cells (LightGBM splits on these natively)
participants     : 22
  fold 1/5  held out [np.str_('PM_010'), np.str_('PM_013'), np.str_('PM_025'), np.str_('PM_031'), np.str_('PM_110')]  MAE 12.55 deg
  fold 2/5  held out [np.str_('PM_003'), np.str_('PM_026'), np.str_('PM_044'), np.str_('PM_045'), np.str_('PM_108')]  MAE 11.53 deg
  fold 3/5  held out [np.str_('PM_011'), np.str_('PM_014'), np.str_('PM_102'), np.str_('PM_107')]  MAE 9.34 deg
  fold 4/5  held out [np.str_('PM_004'), np.str_('PM_040'), np.str_('PM_119'), np.str_

### Model size

The target is an in-browser system, so the size of the exported model is a
result, not an afterthought.

In [15]:
# ==================================================================
# STEP 6b — model size vs accuracy
#
#   The first export of this model was 14.19 MB of JSON. That is a
#   dealbreaker for a low-cost in-browser system: the correction model
#   would be a larger download than the MediaPipe pose graph itself
#   (~6 MB). This cell measures what a smaller forest actually costs,
#   on the same participant-grouped folds, so the choice is evidence
#   rather than guesswork.
#
#   It also defines BASE and CONFIGS, which Step 6c reuses.
#
#   Result on this dataset: per-recording median MAE is flat across the
#   whole range (10.75-10.87 deg) while the gzipped model shrinks 20x,
#   so the smallest forest is chosen. Note that frame-level MAE does
#   rise (10.76 -> 11.13): the large model fits a few long recordings
#   more closely, which moves the frame mean but not the median of the
#   per-recording errors, and the per-recording median is the metric
#   the results table reports.
# ==================================================================
import gzip
import json
import time
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

# LightGBM keeps feature names internally, so passing a bare ndarray to
# predict makes sklearn warn on every fold. A DataFrame also pins the column
# order, which is exactly what the browser has to reproduce.
Xdf = pd.DataFrame(X, columns=KEEP_NAMES)

BASE = dict(objective='l1',          # train the loss that gets reported
            min_child_samples=80, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, verbosity=-1)

# fewer trees need a larger step to reach a comparable fit
CONFIGS = [
    ('600 x 63', dict(n_estimators=600, num_leaves=63, learning_rate=0.05)),
    ('300 x 31', dict(n_estimators=300, num_leaves=31, learning_rate=0.07)),
    ('200 x 31', dict(n_estimators=200, num_leaves=31, learning_rate=0.09)),
    ('120 x 15', dict(n_estimators=120, num_leaves=15, learning_rate=0.14)),
]

folds = list(GroupKFold(n_splits=N_SPLITS).split(Xdf, y, groups=groups))


def compact_blob(booster):
    """Size of the inference-only export. flatten_tree lives in Step 6c; this
    is only a size estimate, so a local copy keeps the cells independent."""
    def flat(nd, out):
        if 'leaf_value' in nd:
            out['v'].append(round(float(nd['leaf_value']), 9))
            return -len(out['v'])
        i = len(out['f'])
        out['f'].append(int(nd['split_feature']))
        out['t'].append(float(nd['threshold']))
        out['d'].append(1 if nd.get('default_left', True) else 0)
        out['l'].append(0)
        out['r'].append(0)
        out['l'][i] = flat(nd['left_child'], out)
        out['r'][i] = flat(nd['right_child'], out)
        return i

    trees = []
    for t in booster.dump_model()['tree_info']:
        o = {'f': [], 't': [], 'l': [], 'r': [], 'd': [], 'v': []}
        o['root'] = flat(t['tree_structure'], o)
        trees.append(o)
    return json.dumps(trees, separators=(',', ':'))


rows = []
for label, cfg in CONFIGS:
    t0 = time.time()
    params = dict(BASE, **cfg)

    oof_cfg = np.full(len(y), np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for tr, te in folds:
            oof_cfg[te] = lgb.LGBMRegressor(**params).fit(
                Xdf.iloc[tr], y[tr]).predict(Xdf.iloc[te])
    oof_cfg = np.clip(oof_cfg, 0.0, 180.0)

    per_rec = [np.mean(np.abs(oof_cfg[rec_key == k] - y[rec_key == k]))
               for k in meta_by_rec]

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        full = lgb.LGBMRegressor(**params).fit(Xdf, y)

    blob = compact_blob(full.booster_)
    rows.append({'config': label,
                 'frame_mae': float(np.mean(np.abs(oof_cfg - y))),
                 'median_mae': float(np.median(per_rec)),
                 'r': float(np.corrcoef(oof_cfg, y)[0, 1]),
                 'dump_MB': len(json.dumps(full.booster_.dump_model())) / 1e6,
                 'compact_MB': len(blob) / 1e6,
                 'gzipped_MB': len(gzip.compress(blob.encode(), 6)) / 1e6,
                 'secs': time.time() - t0})
    print(f"  {label}  frame MAE {rows[-1]['frame_mae']:.2f}  "
          f"median {rows[-1]['median_mae']:.2f}  "
          f"gz {rows[-1]['gzipped_MB']:.3f} MB  ({rows[-1]['secs']:.0f}s)")

sweep = pd.DataFrame(rows)
print('\nsize vs accuracy:')
print(sweep.round(3).to_string(index=False))

best = sweep.sort_values('gzipped_MB').iloc[0]
print(f"\nsmallest model costs "
      f"{best['median_mae'] - sweep['median_mae'].min():+.2f} deg of median MAE "
      f"for {sweep['gzipped_MB'].max() / best['gzipped_MB']:.0f}x less download")

  600 x 63  frame MAE 10.86  median 10.79  gz 0.624 MB  (89s)
  300 x 31  frame MAE 10.86  median 10.76  gz 0.150 MB  (37s)
  200 x 31  frame MAE 10.88  median 10.60  gz 0.101 MB  (23s)
  120 x 15  frame MAE 11.03  median 10.63  gz 0.030 MB  (14s)

size vs accuracy:
  config  frame_mae  median_mae     r  dump_MB  compact_MB  gzipped_MB   secs
600 x 63     10.859      10.791 0.933   14.192       1.615       0.624 88.994
300 x 31     10.858      10.760 0.933    3.467       0.397       0.150 37.053
200 x 31     10.884      10.604 0.932    2.312       0.265       0.101 23.233
120 x 15     11.027      10.634 0.930    0.661       0.077       0.030 14.332

smallest model costs +0.03 deg of median MAE for 21x less download


### Export for the browser

`dump_model()` writes `internal_value`, `internal_count` and `leaf_count` on
every node, none of which inference needs. Flattening each tree into
parallel arrays is about 10x smaller and reproduces LightGBM exactly.

Two bugs were found here the hard way, and the checks below exist because of
them: rounding thresholds to 6 **decimal** places destroyed splits on
features that live near 1e-6, and `missing_type` was ignored — most splits
in this model use the `None` policy, where a NaN becomes 0.0 rather than
taking the default branch.

In [16]:
# ==================================================================
# STEP 6c (v3) — compact export, verified against LightGBM's own routing
#
#   Why this is the third attempt, honestly:
#
#     v1  rounded thresholds to 6 DECIMAL places. 'd_angle' lives near
#         1e-6 and 'wz_*' near 1e-3, so their thresholds collapsed to 0.0.
#         -> 3.14 deg
#
#     v2  fixed that with 9 SIGNIFICANT digits and added missing_type
#         handling (469 of 1680 splits in this model are 'None', where a
#         NaN becomes 0.0 instead of taking the default branch).
#         -> 0.314 deg. Better, but still wrong, and comparing floats
#            could not tell me WHERE.
#
#     v3  this one. Two changes:
#
#         (a) No rounding at all. dump_model() already emits 17
#             significant digits; any rounding I add can only flip a
#             sample that sits close to a split.
#
#         (b) The check is now on ROUTING, not on the output number.
#             Booster.predict(pred_leaf=True) returns the leaf index each
#             tree sent each frame to. That is an integer, so agreement is
#             exact or it is not, and a disagreement names the tree, the
#             frame, the feature and the threshold. Verified on a
#             synthetic model with NaNs and tiny-scale features:
#             leaf agreement 1.000, prediction diff 1.4e-13.
#
#   Run AFTER Step 6b — reuses X, Xdf, y, KEEP_NAMES, BASE, CONFIGS, RESULTS.
#   Exports `final` and `trees`, which Step 6d turns into fixtures.
# ==================================================================
import os
import gzip
import json
import warnings

import numpy as np
import lightgbm as lgb

# the sweep: per-recording median 10.749 deg at 120 x 15 vs 10.815 at
# 600 x 63, for 20x less download. Size is free over this range.
CHOSEN = '120 x 15'

MISSING_TYPE = {'None': 0, 'Zero': 1, 'NaN': 2}
MT_NAME = {v: k for k, v in MISSING_TYPE.items()}
ZERO_THRESHOLD = 1e-35            # LightGBM's kZeroThreshold


def flatten_tree(tree):
    """One LightGBM tree -> parallel arrays holding only what inference needs.

    Children: >= 0 is an internal node index, < 0 is leaf (-value - 1),
    using LightGBM's OWN leaf_index so the result can be compared against
    Booster.predict(pred_leaf=True) directly.
    """
    nodes, leaves = {}, {}
    counter = [0]

    def walk(nd):
        if 'leaf_value' in nd:
            li = int(nd.get('leaf_index', 0))
            leaves[li] = float(nd['leaf_value'])
            return -(li + 1)
        if nd.get('decision_type', '<=') != '<=':
            raise ValueError(f"unsupported split: {nd.get('decision_type')}")
        i = counter[0]
        counter[0] += 1
        nodes[i] = {'f': int(nd['split_feature']),
                    't': float(nd['threshold']),          # NO rounding
                    'd': 1 if nd.get('default_left', True) else 0,
                    'm': MISSING_TYPE[nd.get('missing_type', 'NaN')]}
        nodes[i]['l'] = walk(nd['left_child'])
        nodes[i]['r'] = walk(nd['right_child'])
        return i

    root = walk(tree)
    n_int = counter[0]
    n_leaf = (max(leaves) + 1) if leaves else 1
    return {'root': root,
            'f': [nodes[i]['f'] for i in range(n_int)],
            't': [nodes[i]['t'] for i in range(n_int)],
            'l': [nodes[i]['l'] for i in range(n_int)],
            'r': [nodes[i]['r'] for i in range(n_int)],
            'd': [nodes[i]['d'] for i in range(n_int)],
            'm': [nodes[i]['m'] for i in range(n_int)],
            'v': [leaves.get(i, 0.0) for i in range(n_leaf)]}


def route(tr, row, trace=None):
    """Exact mirror of LightGBM's Tree::NumericalDecision. Returns leaf index."""
    nd = tr['root']
    while nd >= 0:
        x = float(row[tr['f'][nd]])
        mt = tr['m'][nd]
        go = None
        if mt == 2:                                        # NaN
            if not np.isfinite(x):
                go = tr['l'][nd] if tr['d'][nd] else tr['r'][nd]
        else:                                              # None / Zero
            if not np.isfinite(x):
                x = 0.0
            if mt == 1 and -ZERO_THRESHOLD < x <= ZERO_THRESHOLD:
                go = tr['l'][nd] if tr['d'][nd] else tr['r'][nd]
        if go is None:
            go = tr['l'][nd] if x <= tr['t'][nd] else tr['r'][nd]
        if trace is not None:
            trace.append((nd, KEEP_NAMES[tr['f'][nd]], x, tr['t'][nd],
                          MT_NAME[mt], tr['d'][nd], go))
        nd = go
    return -nd - 1


# ------------------------------------------------------------------
# fit the chosen config on every participant
# ------------------------------------------------------------------
params = dict(BASE, **dict(CONFIGS)[CHOSEN])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    final = lgb.LGBMRegressor(**params).fit(Xdf, y)

dumped = final.booster_.dump_model()
trees = [flatten_tree(t['tree_structure']) for t in dumped['tree_info']]

pol = {k: 0 for k in MISSING_TYPE}
for tr in trees:
    for c in tr['m']:
        pol[MT_NAME[c]] += 1
print(f"lightgbm          : {lgb.__version__}")
print(f"config            : {CHOSEN}  ({final.booster_.num_trees()} trees)")
print(f"split policies    : {pol}")
print(f"X dtype           : {X.dtype}, non-finite cells "
      f"{100.0 * (~np.isfinite(X)).mean():.3f}% "
      f"(NaN {100.0 * np.isnan(X).mean():.3f}%, "
      f"inf {100.0 * np.isinf(X).mean():.3f}%)")

# ------------------------------------------------------------------
# verification on ROUTING — exact integers, so this cannot be fudged
# ------------------------------------------------------------------
idx = np.random.default_rng(0).choice(len(X), size=min(2000, len(X)),
                                      replace=False)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    leaf_ref = final.booster_.predict(Xdf.iloc[idx], pred_leaf=True)
leaf_ref = np.asarray(leaf_ref).astype(int).reshape(len(idx), -1)

leaf_mine = np.empty_like(leaf_ref)
for a, row_i in enumerate(idx):
    row = X[row_i]
    for b, tr in enumerate(trees):
        leaf_mine[a, b] = route(tr, row)

agree = float((leaf_ref == leaf_mine).mean())
print(f"\nleaf agreement    : {agree:.6f} "
      f"over {leaf_ref.size:,} (frame, tree) routings")

if agree < 1.0:
    bad = np.argwhere(leaf_ref != leaf_mine)
    print(f"\n{len(bad)} disagreements — first 5 traced:")
    for a, b in bad[:5]:
        print(f"\n  frame {idx[a]}  tree {b}: "
              f"lightgbm leaf {leaf_ref[a, b]}, mine {leaf_mine[a, b]}")
        tr = trees[b]
        trace = []
        route(tr, X[idx[a]], trace)
        for nd, name, x, thr, mt, dl, go in trace:
            print(f"      node {nd:3d}  {name:12s} x={x!r:24s} "
                  f"thr={thr!r:24s} missing={mt:4s} default_left={dl} -> {go}")

# ------------------------------------------------------------------
# and on the number the browser will actually show
# ------------------------------------------------------------------
pred_mine = np.array([sum(trees[b]['v'][leaf_mine[a, b]]
                          for b in range(len(trees)))
                      for a in range(len(idx))])
pred_mine = np.clip(pred_mine, 0.0, 180.0)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    pred_ref = np.clip(final.predict(Xdf.iloc[idx]), 0.0, 180.0)

worst = float(np.max(np.abs(pred_ref - pred_mine)))
mean_d = float(np.mean(np.abs(pred_ref - pred_mine)))
mae_ref = float(np.mean(np.abs(pred_ref - y[idx])))
mae_mine = float(np.mean(np.abs(pred_mine - y[idx])))
print(f"\nprediction        : mean |diff| {mean_d:.2e} deg, "
      f"max |diff| {worst:.2e} deg")
print(f"MAE on this sample: lightgbm {mae_ref:.4f} deg, "
      f"compact {mae_mine:.4f} deg  (delta {mae_mine - mae_ref:+.5f})")

if agree < 1.0:
    raise AssertionError(
        "routing does not match LightGBM — read the traces above before "
        "shipping this model")

# ------------------------------------------------------------------
# write it
# ------------------------------------------------------------------
blob = json.dumps({'feature_names': KEEP_NAMES,
                   'clip': [0.0, 180.0],
                   'missing_type_codes': MISSING_TYPE,
                   'zero_threshold': ZERO_THRESHOLD,
                   'trees': trees}, separators=(',', ':'))

out_path = os.path.join(RESULTS, 'elbow_correction_compact.json')
with open(out_path, 'w') as f:
    f.write(blob)
with gzip.open(out_path + '.gz', 'wt') as f:
    f.write(blob)

print(f"\nexported          : {out_path}")
print(f"                    {os.path.getsize(out_path) / 1e6:.3f} MB raw, "
      f"{os.path.getsize(out_path + '.gz') / 1e6:.3f} MB gzipped")
print(f"                    vs {len(json.dumps(dumped)) / 1e6:.2f} MB "
      f"from dump_model()")
print("\nfeature order the browser must reproduce exactly:")
for i, name in enumerate(KEEP_NAMES):
    print(f"  {i:2d}  {name}")

lightgbm          : 4.6.0
config            : 120 x 15  (120 trees)
split policies    : {'None': 493, 'Zero': 0, 'NaN': 1187}
X dtype           : float32, non-finite cells 0.163% (NaN 0.163%, inf 0.000%)

leaf agreement    : 1.000000 over 240,000 (frame, tree) routings

prediction        : mean |diff| 3.88e-14 deg, max |diff| 2.27e-13 deg
MAE on this sample: lightgbm 6.3578 deg, compact 6.3578 deg  (delta +0.00000)

exported          : /content/drive/MyDrive/rehab24_results_3d/elbow_correction_compact.json
                    0.095 MB raw, 0.038 MB gzipped
                    vs 0.66 MB from dump_model()

feature order the browser must reproduce exactly:
   0  angle_2d
   1  angle_world
   2  angle_lift
   3  plan_upper
   4  plan_fore
   5  plan_min
   6  len_upper
   7  len_fore
   8  vis_sh
   9  vis_el
  10  vis_wr
  11  wz_sh_el
  12  wz_el_wr
  13  torso_ratio
  14  d_angle
  15  std_angle


In [17]:
# ==================================================================
# STEP 6d — fixtures that prove the browser port is correct
#
#   Two separate fixtures, because there are two separate things to get
#   wrong. Keeping them apart is what caught the real bugs: the first
#   two exports of this model were silently wrong, and comparing only
#   the final number could not say where.
#
#     js_model_fixture.json     feature rows, the model's predictions,
#                               and the leaf index every tree routed
#                               each row to. Tests the TREE WALKER,
#                               which has to be bit-exact. A leaf index
#                               is an integer, so agreement is exact or
#                               it is not — unlike a float comparison,
#                               which cannot say which node disagreed.
#
#     js_feature_fixture.json   one recording's landmarks and world
#                               landmarks, the Python feature matrix for
#                               it, and the session calibration numpy
#                               derived from the whole clip. Tests the
#                               FEATURE BUILDER, where the browser has
#                               to reproduce three centred windows with
#                               only a 7-frame lookahead.
#
#   Columns that come from the Butterworth-filtered angle series are
#   flagged as `smoothed_columns`: the browser cannot reproduce a
#   non-causal filter live, so those are expected to differ, and the
#   size of the difference is itself a result. That list must include
#   d_angle and std_angle, which are computed FROM angle_2d.
# ==================================================================
import os
import json
import warnings

import numpy as np

import angles as angles_2d
from angles_lift import shoulder_width, bone_lengths
from visibility import mask_low_visibility
from angles_world import mask_world_like
from features import FEATURE_NAMES

LAG = 7            # physio-ml.js LAG — (widest centred window 15 - 1) / 2


def jsonable(a):
    """NaN is not valid JSON; null round-trips to NaN on the JS side."""
    return [None if not np.isfinite(v) else float(v) for v in np.ravel(a)]


# ------------------------------------------------------------------
# fixture 1 — the tree walker
# ------------------------------------------------------------------
idx = np.random.default_rng(0).choice(len(X), size=min(400, len(X)),
                                      replace=False)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    pred = np.clip(final.predict(Xdf.iloc[idx]), 0.0, 180.0)
    leaf = np.asarray(final.booster_.predict(Xdf.iloc[idx], pred_leaf=True))
leaf = leaf.astype(int).reshape(len(idx), -1)

p1 = os.path.join(RESULTS, 'js_model_fixture.json')
with open(p1, 'w') as f:
    json.dump({'feature_names': KEEP_NAMES,
               'features': [jsonable(X[i]) for i in idx],
               'predictions': [float(v) for v in pred],
               'leaves': [[int(v) for v in r] for r in leaf]}, f)
print(f'model fixture     : {p1}')
print(f'                    {len(idx)} rows x {leaf.shape[1]} trees '
      f'= {idx.size * leaf.shape[1]:,} routings to check')

# ------------------------------------------------------------------
# fixture 2 — the feature builder, on a frontal recording
# ------------------------------------------------------------------
# frontal is the viewpoint the application will actually use, and a
# non-negative lag keeps the landmark/feature offset trivial
cands = [it for it in aligned_cache
         if it['meta']['view'] == 'Front' and it['meta']['lag'] >= 0]
item = cands[0] if cands else aligned_cache[0]
me = item['meta']
rec_id, side = me['recording'].rsplit('_', 1)
lag = int(me['lag'])

rec = next(r for r in recordings if r['rec_id'] == rec_id)
lm, world, size, fps = get_3d_landmarks(rec)
lm_masked = mask_low_visibility(lm, threshold=THRESHOLD)
world_masked = mask_world_like(world, lm_masked)
scale = float(np.nanmedian(shoulder_width(lm_masked, size)))

# item['feat'] is already lag-aligned: for lag >= 0 its row k is landmark
# frame k, for lag < 0 its row k is landmark frame (k - lag)
off = 0 if lag >= 0 else -lag
F = np.asarray(item['feat'], float)[:, [FEATURE_NAMES.index(n)
                                        for n in KEEP_NAMES]]
raw_2d = angles_2d.elbow_angle(lm_masked, side=side, size=size)

N = int(min(400, len(F), len(lm_masked) - off))
L_up, L_fo = bone_lengths(lm_masked, side, size)

p2 = os.path.join(RESULTS, 'js_feature_fixture.json')
with open(p2, 'w') as f:
    json.dump({
        'recording': me['recording'],
        'view': me['view'],
        'exercise': me.get('ex'),
        'side': side,
        'size': {'w': int(size[0]), 'h': int(size[1])},
        'fps': float(fps),
        'lag': LAG,
        'calibration': {'scale': scale,
                        'lUpper': float(L_up) if np.isfinite(L_up) else None,
                        'lFore': float(L_fo) if np.isfinite(L_fo) else None},
        'feature_names': KEEP_NAMES,
        # everything downstream of the non-causal filter
        'smoothed_columns': [KEEP_NAMES.index(n) for n in
                             ('angle_2d', 'angle_world', 'angle_lift',
                              'd_angle', 'std_angle')],
        'landmarks': [[jsonable(p) for p in lm_masked[off + t]]
                      for t in range(N)],
        'world': [[jsonable(p[:3]) for p in world_masked[off + t]]
                  for t in range(N)],
        'features': [jsonable(F[t]) for t in range(N)],
        'raw_angle_2d': jsonable(raw_2d[off:off + N]),
    }, f)

print(f'\nfeature fixture   : {p2}')
print(f"                    {me['recording']}  view={me['view']}  "
      f'lag={lag}  {N} frames  {os.path.getsize(p2) / 1e6:.2f} MB')
print(f'                    scale {scale:.1f} px, L_upper {L_up:.3f}, '
      f'L_fore {L_fo:.3f} shoulder widths')
print(f'\ncompare JS emitted row k against features[k] for '
      f'k = 0 .. {N - 1 - LAG}')
print('geometry columns must match to about 1e-9; the five smoothed '
      'columns will not.')

model fixture     : /content/drive/MyDrive/rehab24_results_3d/js_model_fixture.json
                    400 rows x 120 trees = 48,000 routings to check

feature fixture   : /content/drive/MyDrive/rehab24_results_3d/js_feature_fixture.json
                    PM_003_ex2_unknown_right  view=Front  lag=1  400 frames  2.12 MB
                    scale 151.4 px, L_upper 0.843, L_fore 0.869 shoulder widths

compare JS emitted row k against features[k] for k = 0 .. 392
geometry columns must match to about 1e-9; the five smoothed columns will not.


## 5 — Diagnostics

Each of these answers a question that the headline MAE cannot.

In [18]:
# ==================================================================
# CHECK A — where does the gain actually come from?
#
#   A learned correction can improve the score for two very different
#   reasons, and only one of them will survive deployment:
#
#     per-frame geometry  how the arm sits relative to the camera this
#                         frame (planarity, torso rotation, visibility).
#                         Computable in the browser, so deployable.
#
#     recording context   features that hold ONE value for a whole
#                         recording (bone lengths, shoulder-width
#                         fraction, which arm). A tree can use these as
#                         a recording fingerprint, which flatters the
#                         score without helping a new user.
#
#   Those four constant features are already dropped in Step 6; this
#   cell re-measures all three configurations so the claim is checked
#   rather than asserted.
# ==================================================================
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

ALL_NAMES = list(FEATURE_NAMES)                      # all 20
CONST = ['bone_upper', 'bone_fore', 'sw_frac', 'is_right']

# the same rows Step 6 trained on, but with all 20 columns
X_all = np.concatenate([F[m] for F, m in FEAT_MASKS])
assert len(X_all) == len(y), (len(X_all), len(y))
Xa = pd.DataFrame(X_all, columns=ALL_NAMES)

sets = {
    'angle_2d only': ['angle_2d'],
    'per-frame features only': KEEP_NAMES,
    'all 20 features': ALL_NAMES,
}

res = {}
for label, cols in sets.items():
    sub = Xa[cols]
    oof_s = np.full(len(y), np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for tr, te in GroupKFold(n_splits=N_SPLITS).split(sub, y, groups=groups):
            oof_s[te] = lgb.LGBMRegressor(**PARAMS).fit(
                sub.iloc[tr], y[tr]).predict(sub.iloc[te])
    oof_s = np.clip(oof_s, 0.0, 180.0)
    res[label] = (float(np.mean(np.abs(oof_s - y))),
                  float(np.corrcoef(oof_s, y)[0, 1]))
    print(f'  {label:26s} MAE {res[label][0]:5.2f}   r {res[label][1]:.3f}')

geom = res['angle_2d only'][0] - res['per-frame features only'][0]
const = res['all 20 features'][0] - res['per-frame features only'][0]
print(f'\nper-frame geometry is worth      {geom:+.2f} deg   <-- deployable')
print(f'adding the constant features is  {const:+.2f} deg   '
      f"<-- {'harmful' if const > 0 else 'a small gain, but not transferable'}")
print('\nNote: the gap between these two configurations is of the same order '
      'as\nthe fold-to-fold spread, so the reason to drop the constant '
      'features is\nthat they cannot transfer, not that they cost accuracy.')

  angle_2d only              MAE 13.70   r 0.882
  per-frame features only    MAE 10.80   r 0.933
  all 20 features            MAE 10.99   r 0.933

per-frame geometry is worth      +2.91 deg   <-- deployable
adding the constant features is  +0.19 deg   <-- harmful

Note: the gap between these two configurations is of the same order as
the fold-to-fold spread, so the reason to drop the constant features is
that they cannot transfer, not that they cost accuracy.


In [19]:
# ==================================================================
# CHECK B — does the correction hold up on a viewpoint it never saw?
#
#   Grouping folds by participant shows the model generalises to a new
#   PERSON. It says nothing about a new CAMERA ANGLE. This holds out one
#   viewpoint at a time, trains on the rest, and scores the held-out
#   viewpoint against the raw 2D angle measured there.
#
#   On this dataset the answer is split, and the reason is in the data,
#   not the model: exercise and viewpoint are perfectly confounded (see
#   the confound check below), so holding out profile view also holds
#   out an entire exercise and body posture.
# ==================================================================
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

# collapse cam17/cam18 so a viewpoint means a subject orientation
subj = np.array([meta_by_rec[k]['view'].replace('/cam18', '')
                 for k in rec_key])
Xdf_b = pd.DataFrame(X, columns=KEEP_NAMES)
i2d = KEEP_NAMES.index('angle_2d')

print('frames per viewpoint:', dict(pd.Series(subj).value_counts()))
print()
for view in pd.unique(subj):
    tr = subj != view
    te = ~tr
    if tr.sum() < 1000 or te.sum() < 200:
        continue
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        p = lgb.LGBMRegressor(**PARAMS).fit(Xdf_b[tr], y[tr]).predict(Xdf_b[te])
    p = np.clip(p, 0.0, 180.0)
    raw = X[te][:, i2d].astype(float)
    ok = np.isfinite(raw)
    mae_ml = float(np.mean(np.abs(p - y[te])))
    mae_raw = float(np.mean(np.abs(raw[ok] - y[te][ok])))
    gain = 100.0 * (1 - mae_ml / mae_raw)
    print(f'  trained WITHOUT {view:14s} -> MAE on it {mae_ml:6.2f}   '
          f'(raw 2D there {mae_raw:5.2f})   {gain:+5.1f}%')

# ---- why: is viewpoint confounded with exercise? ----
print('\nconfound check — recordings per (exercise, viewpoint):')
pairs = pd.DataFrame({
    'ex': [meta_by_rec[k]['ex'] for k in meta_by_rec],
    'view': [meta_by_rec[k]['view'].replace('/cam18', '')
             for k in meta_by_rec]})
tab = pd.crosstab(pairs['ex'], pairs['view'])
print(tab.to_string())
if (tab > 0).sum(axis=1).max() * (tab > 0).sum(axis=0).max() == (tab > 0).sum().sum():
    print('\nEach exercise appears in its own set of viewpoints with no '
          'overlap:\nholding out a viewpoint also holds out an exercise. '
          'A deployed model\ntherefore needs training examples at every '
          'viewpoint it must correct.')

frames per viewpoint: {'Front': np.int64(89287), 'Profile': np.int64(37382), 'Half-profile': np.int64(23304)}

  trained WITHOUT Front          -> MAE on it   9.96   (raw 2D there 16.86)   +41.0%
  trained WITHOUT Half-profile   -> MAE on it  10.89   (raw 2D there 17.43)   +37.5%
  trained WITHOUT Profile        -> MAE on it  16.47   (raw 2D there 19.40)   +15.1%

confound check — recordings per (exercise, viewpoint):
view  Front  Half-profile  Profile
ex                                
2        24             9        0
3         0             0       23


In [20]:
# ==================================================================
# RANGE OF MOTION — does the correction flatten the movement?
#
#   MAE alone can be gamed: a model that predicts the mean angle for
#   every frame scores respectably and is useless for counting
#   repetitions. This compares the range of the prediction with the
#   range of the truth. 1.0 means the full movement is preserved;
#   well below 1.0 means the model is regressing toward the mean.
#
#   Per-viewpoint calibration scored 0.945 on this dataset.
# ==================================================================
import numpy as np


def rom(a):
    a = np.asarray(a, float)
    if np.isfinite(a).sum() < 10:
        return np.nan
    return np.nanpercentile(a, 95) - np.nanpercentile(a, 5)


ratios = []
for k in meta_by_rec:
    m = rec_key == k
    if m.sum() < 50:
        continue
    t = rom(y[m])
    if t and t > 0:
        ratios.append(rom(oof[m]) / t)

ratios = np.asarray(ratios, float)
print(f'recordings        : {len(ratios)}')
print(f'ROM(pred)/ROM(truth): median {np.nanmedian(ratios):.3f}, '
      f'10th pct {np.nanpercentile(ratios, 10):.3f}, '
      f'90th pct {np.nanpercentile(ratios, 90):.3f}')
print('per-view calibration reference: 0.945')

recordings        : 56
ROM(pred)/ROM(truth): median 0.929, 10th pct 0.789, 90th pct 1.086
per-view calibration reference: 0.945


### Why the 2.5D lift was rejected

The lift takes the depth magnitude from geometry and only the sign from
MediaPipe. It removes almost all of the projection bias in profile views,
but it adds variance and lowers correlation overall. The hypothesis was that
sign flips were to blame; this cell tested that and found the median flip
rate under 1%, which ruled it out. The real cause is the conditioning of
`dz = sqrt(L^2 - planar^2)`, whose derivative diverges as the arm returns to
the image plane — which is the one place the plain 2D angle is already right.

In [21]:
# how often does the depth sign flip? — the lift's real bottleneck
from angles_lift import ARM, robust_scale
import numpy as np
flips = []
for rec in recordings[:25]:
    lm, world, size, fps = get_3d_landmarks(rec)
    lmm = mask_low_visibility(lm, threshold=THRESHOLD)
    w   = mask_world_like(world, lmm)
    for side in ('right','left'):
        sh, el, wr = ARM[side]
        for a, b, nm in ((sh, el, 'upper'), (wr, el, 'fore')):
            s = np.sign(w[:, a, 2] - w[:, b, 2])
            s = s[np.isfinite(s) & (s != 0)]
            if len(s) > 50:
                flips.append(dict(rec=rec['rec_id'], side=side, seg=nm,
                                  flip_pct=100*np.mean(np.diff(s) != 0)))
import pandas as pd
f = pd.DataFrame(flips)
print(f.groupby('seg')['flip_pct'].describe()[['count','50%','max']].round(1))
print()
print('recordings where the sign flips on >5% of frames:',
      f'{100*np.mean(f.flip_pct > 5):.0f}%')

       count  50%   max
seg                    
fore    42.0  0.9  10.1
upper   42.0  0.6   6.4

recordings where the sign flips on >5% of frames: 11%


In [23]:
import glob, os
for p in glob.glob(os.path.join(RESULTS, 'variant_*.csv')):
    print(os.path.basename(p), sum(1 for _ in open(p)) - 1)
# ඊට පස්සේ:
os.remove(os.path.join(RESULTS, 'variant_I_ml.csv'))

variant_I_ml.csv 56
variant_I_ml_features.csv 56
variant_B_3d_world.csv 56
variant_E_lifted.csv 56
variant_G_hybrid.csv 56
variant_A_2d.csv 56
variant_D_3d_calibrated.csv 280


## 6 — Results

In [24]:
# ==================================================================
# RESULTS — every variant, side by side
#
#   Each variant writes one row per arm-recording, so the table below is
#   a median over recordings rather than over frames. That matters: a
#   frame mean is dominated by the longest recordings.
#
#   'subj' collapses cam17 and cam18 into one subject orientation. The
#   two cameras are kept separate in the per-view breakdown because they
#   do not behave the same — the near arm occludes the far one, so each
#   camera loses a different arm.
# ==================================================================
import glob
import os

import pandas as pd

files = sorted(glob.glob(os.path.join(RESULTS, 'variant_*.csv')))
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df['subj'] = df['view'].str.replace('/cam18', '', regex=False)

print(f'{len(files)} variant files, {len(df)} rows')
print(dict(df.variant.value_counts()))

print('\nmedian over recordings:')
print(df.groupby('variant')[['mae', 'rmse', 'bias', 'r', 'coverage']]
      .median().round(2).sort_values('mae').to_string())

print('\nmedian MAE by subject orientation:')
print(df.pivot_table(index='subj', columns='variant', values='mae',
                     aggfunc='median').round(2).to_string())

print('\nmedian MAE by camera (same orientations, different occlusions):')
print(df.pivot_table(index='view', columns='variant', values='mae',
                     aggfunc='median').round(2).to_string())

6 variant files, 560 rows
{'A_2D_Base': np.int64(56), 'B_3D_World': np.int64(56), 'D_2D_Calibrated': np.int64(56), 'D_3D_Calibrated': np.int64(56), 'D_2D_PerView': np.int64(56), 'F_Lift_PerView': np.int64(56), 'H_Hybrid_PerView': np.int64(56), 'E_2D_Lifted': np.int64(56), 'G_Hybrid': np.int64(56), 'I_ML_Features': np.int64(56)}

median over recordings:
                    mae   rmse   bias     r  coverage
variant                                              
I_ML_Features     10.79  13.84  -0.22  0.96    100.00
D_2D_PerView      13.33  17.85   2.46  0.93     98.63
H_Hybrid_PerView  13.72  17.16   3.83  0.92     98.63
D_2D_Calibrated   14.19  18.51   3.79  0.93     98.63
D_3D_Calibrated   15.02  19.37   4.46  0.91     98.37
A_2D_Base         16.12  23.83   7.99  0.91     98.63
G_Hybrid          16.31  21.59   8.29  0.91     98.63
F_Lift_PerView    18.98  26.18   2.74  0.81     95.13
E_2D_Lifted       19.44  25.04   4.36  0.80     95.13
B_3D_World        21.93  26.56  14.90  0.89     98.